<a href="https://colab.research.google.com/github/shibanidsai/MLOPS/blob/main/Model_Expriment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Installing weight and biases library

In [15]:
!pip install wandb

## Loading the dataset: Used Car Price Prediction

In [16]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
import wandb
import os

In [17]:
from google.colab import files
uploaded = files.upload()

Saving cars.csv to cars (1).csv


In [18]:
cars_df = pd.read_csv( "cars.csv" )

In [19]:
cars_df.head(5)

,Location,Fuel_Type,Transmission,Owner_Type,Seats,Price,age,KM_Driven,make,mileage,engine,power
0,Ahmedabad,Petrol,Manual,First,5.0,3.90,5,53,toyota,17.71,1197,78.90
1,Coimbatore,Petrol,Manual,First,5.0,3.91,2,38,maruti,15.10,1196,73.00
2,Bangalore,Petrol,Manual,First,4.0,2.15,5,25,tata,25.40,624,37.50
3,Coimbatore,Petrol,Automatic,First,5.0,6.55,3,39,nissan,19.15,1198,75.94
4,Mumbai,Diesel,Manual,First,5.0,7.50,3,55,maruti,24.30,1248,88.50


In [20]:
x_columns = ['KM_Driven', 'Fuel_Type', 'age',
             'Transmission', 'Owner_Type', 'Seats',
             'make', 'mileage', 'engine',
             'power', 'Location']
## model of the car is not included in the model

In [21]:
cars_df.shape

(1038, 12)

In [22]:
cars_df = cars_df[x_columns + ['Price']].dropna()

In [23]:
cars_df.shape

(1037, 12)

## Identifying numerical and categorical features

In [24]:
cat_features = ['Fuel_Type',
                'Transmission', 'Owner_Type',
                'make', 'Location']

In [25]:
# set(x_columns) converts all column names into a set to remove duplicates and allow set operations.
num_features = list(set(x_columns) - set(cat_features))


## Utility method for preparing the data

- Splitting the dataset
- Encoding Catgorical Variables

In [26]:
X = cars_df[x_columns]
y = cars_df.Price

In [27]:
# Split the dataset into train and test split
x_train, x_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    train_size = 0.8,
                                                    random_state = 100)

### Creating ML Pipeline

In [28]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [29]:
#ohe_encoder = OneHotEncoder(handle_unknown='ignore')
# Creates a OneHotEncoder object that converts categorical variables into binary columns and ignores unseen categories during prediction instead of throwing an error.

#scaler = StandardScaler()
# Creates a StandardScaler object that standardizes numerical features to have mean 0 and standard deviation 1.

ohe_encoder = OneHotEncoder(handle_unknown='ignore')
scaler = StandardScaler()

## Creating the imputer for columns that have missing values
imputed_num_vars = ['Seats']
non_imputed_num_vars = list(set(num_features) - set(imputed_num_vars))
mean_imputer = SimpleImputer(strategy='mean')


## Pipeline for the applying imputation and then scaling
imputed_num_transformer = Pipeline( steps = [
        ('imputation', mean_imputer),
        ('scaler', scaler)])

non_imputed_num_transformer = Pipeline( steps = [('scaler', scaler)])


## Pipeline for OHE encoding the categorical columns
cat_transformer = Pipeline( steps = [('ohencoder', ohe_encoder)])

## The complete pipeline for applying the required transformatinons to the respective columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num_imputed', imputed_num_transformer, imputed_num_vars),
        ('num_not_imputed', non_imputed_num_transformer, non_imputed_num_vars),
        ('catvars', cat_transformer, cat_features)])

## Initilializing Weights and Biases

In [30]:
os.environ["WANDB_API_KEY"] = "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"

## Baseline Model: Linear Regression

In [31]:
import joblib # Added for saving the model
import os # Added for optional cleanup

linear_reg = LinearRegression()

linear_model = Pipeline(steps=[('preprocessor', preprocessor),
                               ('linear_model', linear_reg)])
## Pipeline for the applying imputation and then scaling

linear_model.fit(x_train, y_train)

wandb.init(project='mlops_usedcar', config=None, tags = ['Linear Model', 'baseline', 'OHE Encoding'])
wandb.run.name = "LinearModel"
rmse = np.sqrt(mean_squared_error(y_test, linear_model.predict(x_test)))
r2 = linear_model.score(x_test, y_test)

wandb.log( {"rmse" : rmse,
            "r2": r2} )

# Save the trained model to a file locally
model_filename = "linear_model.pkl"
joblib.dump(linear_model, model_filename)

# Create a Weights & Biases Artifact
model_artifact = wandb.Artifact(
    "LinearModelArtifact",  # Renamed artifact to be more descriptive
    type = 'model',
    description = 'Linear Regression Model for Used Car Price Prediction'
)

# Add the saved model file to the artifact
model_artifact.add_file(model_filename)

# Log the artifact to Weights & Biases
wandb.log_artifact(model_artifact)

# Finish the W&B run
wandb.finish()

# Optional: Clean up the local model file after logging
os.remove(model_filename)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: shibanidsai (shibanidsai-indian-institute-of-management-bangalore) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


r2,▁
rmse,▁
r2,0.78906
rmse,0.93277


In [32]:
import joblib # Added for saving the model
import os # Added for optional cleanup

params = {"max_depth": 10}

dtree = DecisionTreeRegressor(**params)

dtree_model = Pipeline(steps=[('preprocessor', preprocessor),
                               ('dt_model', dtree)])

dtree_model.fit(x_train, y_train)

wandb.init(project='mlops_usedcar', config=params, tags = ['Decision Tree',
                                                           'OHE Encoding'])
wandb.run.name = "DecisionTree"
rmse = np.sqrt(mean_squared_error(y_test, dtree_model.predict(x_test)))
r2 = dtree_model.score(x_test, y_test)

wandb.log( {"rmse" : rmse,
            "r2": r2} )

# Save the trained model to a file locally
model_filename = "decision_tree_model.pkl"
joblib.dump(dtree_model, model_filename)

# Create a Weights & Biases Artifact
model_artifact = wandb.Artifact(
    "DecisionTreeArtifact",
    type = 'model',
    description = f'Decision Tree Model with max_depth={params["max_depth"]}' # Corrected description to be a string
)

# Add the saved model file to the artifact
model_artifact.add_file(model_filename)

# Log the artifact to Weights & Biases
wandb.log_artifact(model_artifact)

# Finish the W&B run
wandb.finish()

# Optional: Clean up the local model file after logging
os.remove(model_filename)

r2,▁
rmse,▁
r2,0.73586
rmse,1.04378


## Manual Grid Search

In [33]:
from sklearn.model_selection import GridSearchCV

In [34]:
params = { "dt_model__max_depth" : range(5, 10)}

In [35]:
dtree = DecisionTreeRegressor()

dtree_model = Pipeline(steps=[('preprocessor', preprocessor),
                               ('dt_model', dtree)])

In [36]:
dt_grid = GridSearchCV(dtree_model,
                       param_grid = params,
                       cv = 10,
                       scoring = 'r2')

In [37]:
dt_grid.fit(x_train, y_train)

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num_imputed',
                                                                         Pipeline(steps=[('imputation',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Seats']),
                                                                        ('num_not_imputed',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['power',
                                                                          'mileage',
                                                                          'engine',
                                                                          'KM_Driven',
                                                                          'age']),
                                                                        ('catvars',
                                                                         Pipeline(steps=[('ohencoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Fuel_Type',
                                                                          'Transmission',
                                                                          'Owner_Type',
                                                                          'make',
                                                                          'Location'])])),
                                       ('dt_model', DecisionTreeRegressor())]),
             param_grid={'dt_model__max_depth': range(5, 10)}, scoring='r2')

In [38]:
dt_grid.best_params_

{'dt_model__max_depth': 8}

In [39]:
dt_grid.best_score_

np.float64(0.7131063792510498)

In [40]:
pd.DataFrame(dt_grid.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_dt_model__max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.026149,0.011675,0.012755,0.002059,5,{'dt_model__max_depth': 5},0.702526,0.558555,0.657594,0.695006,0.738913,0.697023,0.686102,0.737424,0.714493,0.784153,0.697179,0.056714,5
1,0.022505,0.000782,0.012043,0.000698,6,{'dt_model__max_depth': 6},0.714382,0.649646,0.676392,0.726340,0.725160,0.666815,0.712068,0.777868,0.701987,0.703069,0.705373,0.034149,4
2,0.024346,0.000841,0.011923,0.000234,7,{'dt_model__max_depth': 7},0.686099,0.640435,0.694523,0.722573,0.743101,0.631847,0.733173,0.793476,0.724345,0.713683,0.708326,0.045506,3
3,0.026481,0.002142,0.012955,0.002690,8,{'dt_model__max_depth': 8},0.698660,0.614677,0.759779,0.706717,0.715095,0.598144,0.694505,0.816402,0.760638,0.766446,0.713106,0.064387,1
4,0.027368,0.001512,0.011465,0.000466,9,{'dt_model__max_depth': 9},0.725433,0.652814,0.764980,0.706394,0.699345,0.464484,0.750384,0.820561,0.761561,0.779386,0.712534,0.093915,2


### Using Sweep Features

In [41]:
def train_decision_tree(config=None):
    # Initialize WandB
    with wandb.init(config=config):
        config = wandb.config

        dtree = DecisionTreeRegressor(max_depth=config.max_depth)

        dtree_model = Pipeline(steps=[('preprocessor', preprocessor),
                                      ('dt_model', dtree)])
        dtree_model.fit(x_train, y_train)

        # Evaluate the model
        rmse = np.sqrt(mean_squared_error(y_test, dtree_model.predict(x_test)))
        r2 = dtree_model.score(x_test, y_test)

        # Log metrics to WandB
        wandb.log( {"rmse" : rmse,
                    "r2": r2,
                    "max_depth": config.max_depth} )


In [42]:
sweep_config = {
    "method": "grid",  # Can be 'grid', 'random', or 'bayes'
    "metric": {"name": "r2", "goal": "maximize"},
    "parameters": {
        "max_depth": {
            "values": [4, 6, 8, 12]  # Depths to evaluate
        },
    },
}

In [43]:
sweep_id = wandb.sweep(sweep_config, project="mlops_usedcar")

Create sweep with ID: x3zxh1xl
Sweep URL: https://wandb.ai/shibanidsai-indian-institute-of-management-bangalore/mlops_usedcar/sweeps/x3zxh1xl


In [44]:
wandb.agent(sweep_id,
            function=train_decision_tree)  # Run all experiments

wandb: Agent Starting Run: u8xpxrbf with config:
wandb: 	max_depth: 4
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


max_depth,▁
r2,▁
rmse,▁
max_depth,4
r2,0.66376
rmse,1.17766


wandb: Agent Starting Run: b1z53lkk with config:
wandb: 	max_depth: 6
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


max_depth,▁
r2,▁
rmse,▁
max_depth,6
r2,0.74156
rmse,1.03247


wandb: Agent Starting Run: 7vwjztt0 with config:
wandb: 	max_depth: 8
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


max_depth,▁
r2,▁
rmse,▁
max_depth,8
r2,0.76081
rmse,0.99326


wandb: Agent Starting Run: jpre8j7d with config:
wandb: 	max_depth: 12
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


max_depth,▁
r2,▁
rmse,▁
max_depth,12
r2,0.66615
rmse,1.17347


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


## Get Experiment Details

In [46]:
api = wandb.Api()

all_runs = api.runs("mlops_usedcar", order="+summary_metrics.rmse")

for run in all_runs:
  print(f"Model Name: {run.name} and R2 {run.summary.get('r2')}")
  print(run.config)

Model Name: LinearModel and R2 0.7890613323368604
{}
Model Name: polar-sweep-3 and R2 0.7608141215005836
{'max_depth': 8}
Model Name: hopeful-sweep-2 and R2 0.7415558898903304
{'max_depth': 6}
Model Name: DecisionTree and R2 0.7358623471145105
{'max_depth': 10}
Model Name: kind-sweep-4 and R2 0.6661481322190863
{'max_depth': 12}
Model Name: smart-sweep-1 and R2 0.6637596140757374
{'max_depth': 4}


### Storing the model into a file

In [47]:
from joblib import dump

MODEL_DIR = "./carsmodel"

os.mkdir(MODEL_DIR)
dump(linear_model, MODEL_DIR + "/" + 'cars.pkl')

['./carsmodel/cars.pkl']

### Logging the model artifact in the tracking tools (weights and Biases)

In [48]:
wandb.init(project='mlops_usedcar',
           config=None,
           tags = ['Final Model'])
wandb.run.name = "FinalModel"

In [49]:
model_artifact = wandb.Artifact("Linear_Model_UsedCar",
                                type = 'model',
                                description = 'Linear Model for used car price prediction')

In [50]:
model_artifact.add_dir(MODEL_DIR)

wandb: Adding directory to artifact (carsmodel)... Done. 0.0s


In [51]:
wandb.run.log_artifact(model_artifact)

<Artifact Linear_Model_UsedCar>

In [53]:
wandb.finish()

In [ ]:
import sklearn
sklearn.__version__